In [1]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/usr/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import Prompts as prompts


In [ ]:
import importlib
importlib.reload(prompts)

In [2]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [3]:
from Prompts.prompts_hotels import Config

In [4]:
print(Config.model)

llama3.2:3b


In [5]:
# Load your CSV
df_new = pd.read_csv("/home/sohy47ma/ReviewProject_new/fake_reviews_prediction/Datasets/Hotel_HumanReal_VS_CG.csv")
reviews = df_new["text"].tolist()

In [6]:
prompt = PromptTemplate(
    input_variables=["text"], # the placeholder to be replaced based on prompt template
    template=Config.zero_shot_prompt_template
)

# Load small model
llm = OllamaLLM(model=Config.model)

# Create chain using pipe operator (modern LangChain syntax)
chain = prompt | llm

In [7]:
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a hotel that is listed on a website as either "fake" or "real". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording.

    Review: "{text}"

    Classify this review as either "fake" or "real" (respond with only one word):


In [ ]:
apptainer --version

In [ ]:
import os
os.environ["OLLAMA_HOST"] = "http://localhost:11400"

In [ ]:
# Classify each review
results = []

for r in reviews:
    result = chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        cleaned_label = "fake"
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    results.append(cleaned_label)


In [9]:
df_new.head(10)


,Binary_label,Category,domain,text,is_synthetic,source
0,real,james,Hotel,My husband and I were there for a conference a...,0,Web
1,real,palmer,Hotel,Here are my experiences: - Had three rooms res...,0,Web
2,real,omni,Hotel,My wife and I travelled to Chicago and really ...,0,TripAdvisor
3,real,knickerbocker,Hotel,The location of this hotel is excellent. The h...,0,Web
4,real,fairmont,Hotel,We recently completed our second stay at the F...,0,TripAdvisor
5,real,amalfi,Hotel,The location was good. Check-in was ok. Asked ...,0,Web
6,real,intercontinental,Hotel,After reading some of the most recent reviews ...,0,TripAdvisor
7,real,monaco,Hotel,The service was reasonably well...they seemed ...,0,Web
8,real,hilton,Hotel,I recently stayed here for the Chicago triathl...,0,Web
9,real,ambassador,Hotel,"Stay Away! After generations as an old-world, ...",0,Web


In [10]:
# def clean_label(text):
#     return text.strip().lower().split()[0].strip('.,!?;:')
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [ ]:
y_true = df_new["Binary_label"].tolist()

In [ ]:
for label in results:
    print(label)

In [13]:
accuracy, f1, conf_matrix = evaluate_model(df_new, results)
print(Config.model)
print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS HOTELS - HUMAN REAL  VS COMPUTER GENERATED : LLAMA 3.2 3B")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

llama3.2:3b
ZERO-SHOT PROMPTING RESULTS HOTELS - HUMAN REAL  VS COMPUTER GENERATED : LLAMA 3.2 3B
Accuracy: 0.4949748743718593

F1 Score: 0.3332833283328333

Confusion Matrix:
[[  2 794]
 [ 10 786]]


In [11]:
# Results with LLAMA 3.1 8B
print(Config.model)
accuracy, f1, conf_matrix = evaluate_model(df_new, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS HOTELS - HUMAN REAL  VS COMPUTER GENERATED : LLAMA 3.1 8B")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

llama3.1:8b
ZERO-SHOT PROMPTING RESULTS HOTELS - HUMAN REAL  VS COMPUTER GENERATED : LLAMA 3.1 8B
Accuracy: 0.5244974874371859

F1 Score: 0.4035047949019365

Confusion Matrix:
[[ 59 737]
 [ 20 776]]


## One-Shot Prompting
Using one example to guide the model's classification

In [12]:
llm = OllamaLLM(model=Config.model)

In [13]:
one_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template = Config.one_shot_prompt_template
)
# Create one-shot chain
one_shot_chain = one_shot_prompt | llm

In [14]:
print(Config.one_shot_prompt_template)


     You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a hotel that is listed on a website as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Example:
    Review: "As a former Chicagoan, I'm appalled at the Amalfi Hotel Chicago. First of all, I was expecting luxury and hospitality, neither of which I received. There's an Experience Designer who is supposed to be like a 'personal concierge,' but my experience with my ED was terrible. I felt like he was trying to pressure me into staying more days than I wanted to. Not only that, but I couldn't understand what he was saying most of the time because he was talking so fast. When I finally got to my room, I was disappointed with the quality of the furniture and the room's cleanliness. I had to ask for a maid to come and give me clean towel

In [ ]:
# Classify each review
one_shot_results = []

for r in reviews:
    result = one_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        cleaned_label = "fake"
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    one_shot_results.append(cleaned_label)


In [ ]:
df_new.head(10)

In [11]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [19]:
#One shot results HOTEL HUMAN VS HUMAN FAKE with LLAMA 3.2 3B
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)

print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS HOTEL HUMAN REAL VS COMPUTER GENERATED WITH LLAMA 3.2 3B")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

ONE-SHOT PROMPTING RESULTS HOTEL HUMAN REAL VS COMPUTER GENERATED WITH LLAMA 3.2 3B
Accuracy: 0.5025125628140703

F1 Score: 0.48639722355739856

Confusion Matrix:
[[259 537]
 [255 541]]


In [17]:
#One shot results HOTEL HUMAN VS COMPUTER GENERATED with LLAMA 3.1 8B
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)
print(Config.model)
print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS HOTEL HUMAN REAL VS COMPUTER GENERATED WITH LLAMA 3.1 8B")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

llama3.1:8b
ONE-SHOT PROMPTING RESULTS HOTEL HUMAN REAL VS COMPUTER GENERATED WITH LLAMA 3.1 8B
Accuracy: 0.7399497487437185

F1 Score: 0.739188656577504

Confusion Matrix:
[[632 164]
 [250 546]]


## Few-Shot Prompting
Using multiple examples to guide the model's classification

In [6]:
llm = OllamaLLM(model=Config.model)

In [7]:
few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template=Config.few_shot_prompt_template
)

# Create few-shot chain
few_shot_chain = few_shot_prompt | llm

In [8]:
print(Config.few_shot_prompt_template)


    You are an expert at detecting real and fake reviews.


    Task: Classify the following review of a hotel that is listed on a website as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Here are some examples:

    Examples:
    Example 1:
    Review: "As a former Chicagoan, I'm appalled at the Amalfi Hotel Chicago. First of all, I was expecting luxury and hospitality, neither of which I received. There's an Experience Designer who is supposed to be like a 'personal concierge,' but my experience with my ED was terrible. I felt like he was trying to pressure me into staying more days than I wanted. Not only that, but I couldn't understand what he was saying most of the time because he was talking so fast. When I finally got to my room, I was disappointed with the quality of the furniture and the room's cleanliness. I had to ask 

In [9]:
# Classify each review
few_shot_results = []

for r in reviews:
    result = few_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        cleaned_label = "fake"
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    few_shot_results.append(cleaned_label)


Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: 

In [12]:
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)
print(Config.model)
print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS HUMAN REAL VS COMPUTER GENERATED FOR LLAMA 3.2:3B")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

llama3.2:3b
FEW-SHOT PROMPTING RESULTS HUMAN REAL VS COMPUTER GENERATED FOR LLAMA 3.2:3B
Accuracy: 0.6815326633165829

F1 Score: 0.6727365171835386

Confusion Matrix:
[[673 123]
 [384 412]]


In [24]:
# Results for few shot HUMAN VS COMPUTER GENERATED - llama 3.1 8b
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)
print(Config.model)
print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS HOTEL HUMAN REAL VS COMPUTER GENERATED FOR LLAMA 3.1:8B")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

llama3.1:8b
FEW-SHOT PROMPTING RESULTS HOTEL HUMAN REAL VS COMPUTER GENERATED FOR LLAMA 3.1:8B
Accuracy: 0.7368090452261307

F1 Score: 0.7294246733149582

Confusion Matrix:
[[718  78]
 [341 455]]


## Compare All Approaches
Compare zero-shot, one-shot, and few-shot results

In [ ]:
# Compare all approaches
comparison_df = pd.DataFrame({
    'Approach': ['Zero-Shot', 'One-Shot', 'Few-Shot'],
    'Accuracy': [accuracy, accuracy_one_shot, accuracy_few_shot],
    'F1 Score': [f1, f1_one_shot, f1_few_shot]
})

print("=" * 60)
print("COMPARISON OF ALL PROMPTING APPROACHES")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

# Find best approach
best_idx = comparison_df['Accuracy'].idxmax()
best_approach = comparison_df.loc[best_idx, 'Approach']
best_accuracy = comparison_df.loc[best_idx, 'Accuracy']
print(f"\nBest Approach: {best_approach} (Accuracy: {best_accuracy:.4f})")